In [ ]:
def gradient_projection(A, y, x, tau, maxits, alpha_min, alpha_red, epsilon, c1):

    r = A @ x - y
    f = 0.5 * (r.T @ r)  # Initial function value

    for iter in range(maxits):
        g = A.T @ r  # Gradient of the function
        alpha = 1.0
        stop_flag = 0
        
        while (alpha > alpha_min) and (stop_flag == 0):
            # Projection onto the simplex
            x_new = projsplx(x - alpha * g, tau)
            
            # Check convergence tolerance
            tol_check = np.linalg.norm(x - x_new)
            if (alpha == 1) and (iter > 1):
                if tol_check <= epsilon:
                    stop_flag = 1
            
            # Compute the new residual and function value
            r_new = A @ x_new - y
            f_new = 0.5 * (r_new.T @ r_new)
            
            # Check the Armijo condition (sufficient decrease condition)
            if f_new <= f + c1 * ((x_new - x).T @ g):
                break
            
            alpha *= alpha_red  # Reduce step size
        
        if stop_flag == 1:
            # Successful exit
            err = 0
            return x, err
        
        if alpha < alpha_min:
            # Unsuccessful exit due to too small step length
            print(" *** No suitable step length found ***")
            err = 1
            return x, err
        
        # Update values for next iteration
        r = r_new
        x = x_new
        f = f_new
    
    # If maximum iterations reached
    err = 1
    return x, err

def projsplx(v, tau):
    """
    Projects the vector `v` onto the simplex defined by tau.

    Parameters:
    - v: Vector to project
    - tau: Simplex constraint

    Returns:
    - Projected vector `x`
    """
    # Sort `v` in decreasing order
    u = np.sort(v)[::-1]
    cssv = np.cumsum(u) - tau
    rho = np.nonzero(u > (cssv / np.arange(1, len(v) + 1)))[0][-1]
    theta = cssv[rho] / (rho + 1)
    w = np.maximum(v - theta, 0)
    return w
